In [43]:
'''
Financial Sentiment Classification using FinBERT
Step 3 - Baseline (TF-IDF + Logistic Regression)

Goal: Establish a simple, fast reference score to measure how much
FinBERT actually improves over a traditional text-classification approach.
Not the focus of the project - kept minimal by design.

Input: data/processed/train.csv, val.csv, test.csv (from Step 2).

Model:
- TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=20000) -
  unigrams + bigrams, so short meaningful phrases ("net sales", "fell
  sharply") are captured, not just single words.
- LogisticRegression(class_weight="balanced") - compensates for the class
  imbalance found in EDA (59.4% neutral / 28.1% positive / 12.5% negative),
  so the model isn't biased toward always predicting "neutral".

Evaluation discipline: validation checked first as a sanity check; test
set evaluated only once, for the final number used to compare against
FinBERT later.

Results (test set):
- Accuracy: 78.3%
- Macro F1: 75.4% <- number to beat with FinBERT
- Confusion matrix: neutral is easiest (recall ~84%), negative and positive
  are harder (recall ~70-72%) and are most often confused with neutral -
  expected, since a purely lexical model struggles with mild/implicit
  sentiment and has no understanding of context or negation.

Output: results/baseline_confusion_matrix.png, results/metrics.csv
(FinBERT's results will be appended to the same file for a direct
comparison table).
'''

'\nFinancial Sentiment Classification using FinBERT\nStep 3 - Baseline (TF-IDF + Logistic Regression)\n\nGoal: Establish a simple, fast reference score to measure how much\nFinBERT actually improves over a traditional text-classification approach.\nNot the focus of the project - kept minimal by design.\n\nInput: data/processed/train.csv, val.csv, test.csv (from Step 2).\n\nModel:\n- TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=20000) -\n  unigrams + bigrams, so short meaningful phrases ("net sales", "fell\n  sharply") are captured, not just single words.\n- LogisticRegression(class_weight="balanced") - compensates for the class\n  imbalance found in EDA (59.4% neutral / 28.1% positive / 12.5% negative),\n  so the model isn\'t biased toward always predicting "neutral".\n\nEvaluation discipline: validation checked first as a sanity check; test\nset evaluated only once, for the final number used to compare against\nFinBERT later.\n\nResults (test set):\n- Accuracy: 78.3%\n- M

In [44]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [45]:
PROCESSED_DIR = Path("data/processed")
RESULTS_DIR = Path("results")
LABELS = ["negative", "neutral", "positive"]  # index 0,1,2 - fixed order for reports/plots

In [46]:
def load_splits():
    train = pd.read_csv(PROCESSED_DIR / "train.csv")
    val = pd.read_csv(PROCESSED_DIR / "val.csv")
    test = pd.read_csv(PROCESSED_DIR / "test.csv")
    return train, val, test

In [47]:
def train_baseline(train_df: pd.DataFrame):
    # TF-IDF turns each sentence into a vector of word-importance scores.
    # ngram_range=(1,2) includes both single words and two-word phrases
    # (e.g. "net sales", "fell sharply"), which usually helps sentiment
    # tasks pick up on short meaningful phrases, not just single words.
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    X_train = vectorizer.fit_transform(train_df["sentence"])
    y_train = train_df["label"]
 
    # class_weight="balanced" compensates for the class imbalance we found in
    # EDA (neutral 59%, positive 28%, negative 12.5%) by giving minority-class
    # mistakes a higher penalty during training, instead of letting the model
    # default to always predicting "neutral".
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)
    return vectorizer, clf

In [48]:
def evaluate(vectorizer, clf, df: pd.DataFrame, split_name: str):
    X = vectorizer.transform(df["sentence"])
    y_true = df["label"]
    y_pred = clf.predict(X)
 
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
 
    print(f"\n=== {split_name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))
 
    return y_true, y_pred, {"split": split_name, "accuracy": acc, "macro_f1": macro_f1}

In [49]:
def plot_confusion_matrix(y_true, y_pred, out_path: Path, title: str):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
 

In [50]:
if __name__ == "__main__":
    RESULTS_DIR.mkdir(exist_ok=True)
 
    train, val, test = load_splits()
    vectorizer, clf = train_baseline(train)
 
    # Check validation first - this is the "am I overfitting / is this sane"
    # check. We only look at test once, at the very end, for the number that
    # actually gets compared against FinBERT.
    evaluate(vectorizer, clf, val, "Validation")
    y_true_test, y_pred_test, test_metrics = evaluate(vectorizer, clf, test, "Test")
 
    plot_confusion_matrix(
        y_true_test, y_pred_test,
        RESULTS_DIR / "baseline_confusion_matrix.png",
        "Baseline (TF-IDF + Logistic Regression) - Test Set",
    )
 
    metrics_df = pd.DataFrame([test_metrics])
    metrics_path = RESULTS_DIR / "metrics.csv"
    # Append (don't overwrite) so FinBERT's results land in the same file later,
    # making a side-by-side comparison table trivial.
    if metrics_path.exists():
        existing = pd.read_csv(metrics_path)
        metrics_df = pd.concat([existing, metrics_df], ignore_index=True)
    metrics_df.to_csv(metrics_path, index=False)
 
    print(f"\nSaved confusion matrix and appended metrics to {RESULTS_DIR}/")


=== Validation ===
Accuracy:  0.7624
Macro F1:  0.7020
              precision    recall  f1-score   support

    negative      0.614     0.574     0.593        61
     neutral      0.814     0.854     0.833       287
    positive      0.706     0.654     0.679       136

    accuracy                          0.762       484
   macro avg      0.711     0.694     0.702       484
weighted avg      0.759     0.762     0.760       484


=== Test ===
Accuracy:  0.7831
Macro F1:  0.7540
              precision    recall  f1-score   support

    negative      0.746     0.721     0.733        61
     neutral      0.839     0.836     0.838       287
    positive      0.683     0.699     0.691       136

    accuracy                          0.783       484
   macro avg      0.756     0.752     0.754       484
weighted avg      0.784     0.783     0.783       484


Saved confusion matrix and appended metrics to results/


### OS Path Check

In [51]:
import os
print(os.path.exists("data/processed"))       # checks notebooks/data/processed
print(os.path.exists("../data/processed"))    # checks project-root/data/processed

True
False
